<a href="https://colab.research.google.com/github/neelameghana192311002/CSA6301---Threat-Intelligence-and-Network-Security/blob/main/UNIT4LAB/Exercise_8_Zero_Trust_Continuous_Verification_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
def zero_trust_authorize(request, resource_policy):
    """
    request: {
        "user",
        "mfa_passed",
        "device_posture": {
            "antivirus_enabled",
            "os_patched"
        },
        "resource"
    }

    resource_policy: {resource: set(allowed_users)}

    Access requires:
    - User authorized for the resource
    - MFA completed
    - Healthy device posture

    This is enforced regardless of network location or prior logins.
    """
    reasons = []

    allowed_users = resource_policy.get(request["resource"], set())

    if request["user"] not in allowed_users:
        reasons.append("user not authorized for this resource")

    if not request["mfa_passed"]:
        reasons.append("MFA not completed")

    if not request["device_posture"]["antivirus_enabled"]:
        reasons.append("antivirus disabled")

    if not request["device_posture"]["os_patched"]:
        reasons.append("OS not fully patched")

    return {
        "granted": len(reasons) == 0,
        "reasons": reasons
    }

In [2]:
def test_experiment8():
    policy = {
        "finance_db": {"csmith", "afinance"}
    }

    healthy_request = {
        "user": "csmith",
        "mfa_passed": True,
        "device_posture": {
            "antivirus_enabled": True,
            "os_patched": True
        },
        "resource": "finance_db",
    }

    assert zero_trust_authorize(
        healthy_request,
        policy
    )["granted"] is True

    # Same user, correct password/MFA, but antivirus disabled -> still denied
    unhealthy_request = dict(
        healthy_request,
        device_posture={
            "antivirus_enabled": False,
            "os_patched": True
        }
    )

    result2 = zero_trust_authorize(
        unhealthy_request,
        policy
    )

    assert result2["granted"] is False
    assert "antivirus disabled" in result2["reasons"]

    # A stolen credential for a user never authorized for this resource
    unauthorized_request = dict(
        healthy_request,
        user="attacker99"
    )

    assert zero_trust_authorize(
        unauthorized_request,
        policy
    )["granted"] is False

    print("All test cases passed.")


test_experiment8()

All test cases passed.
